# 08 – NLP Analysis

### Purpose of the Notebook
Analyse textbasierter Vergabefelder mittels NLP.

### Steps
- TF‑IDF preprocessing
- SVD dimensionality reduction
- NMF topic modelling
- SVM text‑risk classifier (3 classes)
- Export TEXT_RISK_SCORE + NLP features
- Integration into modelling pipeline

#### Notes: NLP Pipeline for Tender Risk Prediction
1. Text fields contain additional signals not fully captured by structured variables.  
Tender descriptions often reveal niche requirements, specialized materials, or complex technical specifications that strongly influence the number of bidders and the likelihood of failure.

2. TF‑IDF provides a robust and scalable representation of tender text.  
It captures the importance of domain‑specific terms without requiring deep semantic models, making it suitable for large datasets and classical ML pipelines.

3. Dimensionality reduction (SVD) transforms sparse TF‑IDF vectors into compact numerical features.  
This reduces noise, improves model stability, and allows seamless integration with structured features such as CPV, procedure type, and tender value.

4. Topic modelling (NMF) adds interpretable thematic structure.  
Topics derived from tender descriptions help identify procurement areas with systematically higher failure rates, improving both model interpretability and analytical insights.

5. A text‑based risk classifier (TF‑IDF + SVM) provides a high‑level TEXT_RISK_SCORE.  
The classifier learns patterns associated with failed, medium‑risk, and safe tenders based solely on text, producing a compact categorical feature (low / medium / high) that can be used even when raw text is not available.

6. TEXT_RISK_SCORE is essential for downstream applications such as a risk simulator.  
It allows the model to incorporate text‑derived risk signals without requiring users to input free text, enabling simple scenario simulations based on a limited set of parameters (CPV, procedure type, budget, etc.).

7. The combined pipeline enhances predictive performance while remaining interpretable and scalable.  
TF‑IDF + SVD provides numerical stability, NMF provides thematic insight, and SVM provides a practical risk score — together forming a balanced NLP module that strengthens the overall tender risk prediction model.

--------------------
#### Imports & Setup & Dataset
-------------------

In [1]:
# ---------------------------------------------------------
# Import moduls
# ---------------------------------------------------------


# import standard modules
import pandas as pd
import numpy as np
from pathlib import Path

import sys
from pathlib import Path

In [2]:
# shut off some annoying warnings
import warnings

warnings.filterwarnings("ignore", message="A value is trying to be set on a copy")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# ---------------------------------------------------------
# Setup style
# ---------------------------------------------------------

# show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# visualisation settings
pd.set_option('display.float_format', '{:,.2f}'.format)

In [7]:
# ---------------------------------------------------------
# Load scripts
# ----------------------------------------------------------

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# found min directory
PROJECT_ROOT = Path("..").resolve()

# maindirectory sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# import function from script
from my_scripts.nlp_processing import (preprocess_text, build_tfidf_svd, build_nmf_topics,
                               build_text_risk_classifier, predict_text_risk
)
from my_scripts.eda import overview

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
# ---------------------------------------------------------
# Import dataset
# ---------------------------------------------------------

df = pd.read_pickle("../data/dataset_comp.pkl")
df_de = pd.read_pickle("../data/dataset_de_comp.pkl")

print("EU dataset:", df.shape)
print("DE dataset:", df_de.shape)

EU dataset: (4039906, 37)
DE dataset: (256849, 37)


----------------
### NLP PROCESSING

----------

In [ ]:
# ---------------------------------------------------------
# Preprocess text 
# ---------------------------------------------------------

text_cols = ["TITLE", "SHORT_DESCRIPTION", "DESCRIPTION", "OBJECT_DESCR"]
df = preprocess_text(df, text_cols)


KeyError: "None of [Index(['TITLE', 'SHORT_DESCRIPTION', 'DESCRIPTION', 'OBJECT_DESCR'], dtype='str')] are in the [columns]"

In [10]:
df.columns

Index(['YEAR', 'ID_TYPE', 'XSD_VERSION', 'CANCELLED', 'CORRECTIONS',
       'ISO_COUNTRY_CODE', 'CAE_TYPE', 'B_AWARDED_BY_CENTRAL_BODY',
       'TYPE_OF_CONTRACT', 'TAL_LOCATION_NUTS', 'B_DYN_PURCH_SYST', 'ID_LOT',
       'B_EU_FUNDS', 'TOP_TYPE', 'B_ACCELERATED', 'OUT_OF_DIRECTIVES',
       'CRIT_CODE', 'B_ELECTRONIC_AUCTION', 'NUMBER_AWARDS',
       'B_AWARDED_TO_A_GROUP', 'WIN_COUNTRY_CODE', 'B_CONTRACTOR_SME',
       'B_SUBCONTRACTED', 'AWARD_QUARTER', 'DAYS_TO_AWARD', 'CPV_DIVISION',
       'CPV_GROUP', 'CPV_CLASS', 'IS_FAILED_TENDER', 'IS_LOW_COMPETITION',
       'OFFERS_BIN', 'VALUE_BIN', 'HAS_MULTIPLE_LOTS', 'LOTS_BIN',
       'VALUE_EURO_MISSING', 'AWARD_VALUE_EURO_MISSING',
       'NUMBER_OFFERS_MISSING'],
      dtype='object')

In [ ]:
# ---------------------------------------------------------
# TF‑IDF + SVD features
# ---------------------------------------------------------

df_svd, tfidf_svd, svd_model = build_tfidf_svd(df["TEXT_ALL"])
df = pd.concat([df, df_svd], axis=1)


In [ ]:
# ---------------------------------------------------------
# Topic modelling (NMF)
# ---------------------------------------------------------

df_topics, nmf_model = build_nmf_topics(df["TEXT_ALL"], tfidf_svd)
df = pd.concat([df, df_topics], axis=1)


In [ ]:
# ---------------------------------------------------------
# Prepare labels for TEXT_RISK_SCORE
# ---------------------------------------------------------

risk_map = {
    "failed": "high",
    "low": "medium",
    "medium": "medium",
    "high": "low"
}

df["TEXT_RISK_LABEL"] = df["competition_category"].map(risk_map)


In [ ]:
# ---------------------------------------------------------
# Train SVM classifier
# ---------------------------------------------------------

svm_model, tfidf_svm, label_encoder = build_text_risk_classifier(
    df["TEXT_ALL"],
    df["TEXT_RISK_LABEL"]
)


In [ ]:
# ---------------------------------------------------------
# Predict TEXT_RISK_SCORE
# ---------------------------------------------------------

df["TEXT_RISK_SCORE"] = predict_text_risk(
    df["TEXT_ALL"],
    svm_model,
    tfidf_svm,
    label_encoder
)



---------
### SAVE DATASET

--------

In [ ]:
df.to_pickle("../data/dataset_nlp.pkl")
df_de.to_pickle("../data/dataset_de_nlp.pkl")